[Databricks SQL](https://docs.databricks.com/aws/en/dev-tools/python-sql-connector?language=SQL%C2%A0warehouse#auth-u2m)

In [1]:
import os
from dotenv import load_dotenv
#
load_dotenv()

#
# 1. Configure authentication credentials
# If running inside a Databricks Notebook, the host and token are automatically discovered.
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_PERSONAL_ACCESS_TOKEN")
WAREHOUSE_ID = os.environ.get("DATABRICKS_WAREHOUSE_ID")  # Found under SQL Warehouses -> Connection details


In [ ]:
import os
from sqlalchemy import create_engine

access_token    = os.getenv("DATABRICKS_TOKEN")
server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME")
http_path       = os.getenv("DATABRICKS_HTTP_PATH")
catalog         = os.getenv("DATABRICKS_CATALOG")
schema          = os.getenv("DATABRICKS_SCHEMA")

engine = create_engine(
  url = f"databricks://token:{access_token}@{server_hostname}?" +
        f"http_path={http_path}&catalog={catalog}&schema={schema}"
)

# ...

In [ ]:
from databricks import sql
import os

#
connection = sql.connect(
                server_hostname = DATABRICKS_HOST,
                http_path = f"/sql/1.0/warehouses/{WAREHOUSE_ID}",
                access_token = DATABRICKS_TOKEN
            )

cursor = connection.cursor()

cursor.execute("SELECT * from range(10)")
print(cursor.fetchall())

cursor.close()
connection.close()

[Row(id=0), Row(id=1), Row(id=2), Row(id=3), Row(id=4), Row(id=5), Row(id=6), Row(id=7), Row(id=8), Row(id=9)]


Set User-Agent

In [2]:
from databricks import sql
import os

with sql.connect(server_hostname   = os.getenv("DATABRICKS_HOST"),
                 http_path         = os.getenv("DATABRICKS_HTTP_PATH"),
                 access_token      = os.getenv("DATABRICKS_TOKEN"),
                 user_agent_entry = "product_name") as connection:
  with connection.cursor() as cursor:
    cursor.execute("SELECT 1 + 1")
    result = cursor.fetchall()

    for row in result:
      print(row)



Row((1 + 1)=2)


Query data

In [4]:
from databricks import sql
import os

with sql.connect(server_hostname = os.getenv("DATABRICKS_HOST"),
                 http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
                 access_token    = os.getenv("DATABRICKS_TOKEN")) as connection:

  with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM samples.nyctaxi.trips LIMIT ?", [2])
    result = cursor.fetchall()

    for row in result:
      print(row)

Row(tpep_pickup_datetime=datetime.datetime(2016, 2, 13, 21, 47, 53, tzinfo=<StaticTzInfo 'Etc/UTC'>), tpep_dropoff_datetime=datetime.datetime(2016, 2, 13, 21, 57, 15, tzinfo=<StaticTzInfo 'Etc/UTC'>), trip_distance=1.4, fare_amount=8.0, pickup_zip=10103, dropoff_zip=10110)
Row(tpep_pickup_datetime=datetime.datetime(2016, 2, 13, 18, 29, 9, tzinfo=<StaticTzInfo 'Etc/UTC'>), tpep_dropoff_datetime=datetime.datetime(2016, 2, 13, 18, 37, 23, tzinfo=<StaticTzInfo 'Etc/UTC'>), trip_distance=1.31, fare_amount=7.5, pickup_zip=10023, dropoff_zip=10023)


Query tags

In [5]:
# Session-level tags:

from databricks import sql
import os

with sql.connect(
    server_hostname = os.getenv("DATABRICKS_HOST"),
    http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
    access_token    = os.getenv("DATABRICKS_TOKEN"),
    query_tags = {"team": "engineering", "dashboard": "abc123", "env": "prod"}
) as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT * FROM samples.nyctaxi.trips LIMIT ?", [2])
        result = cursor.fetchall()
        for row in result:
            print(row)

Row(tpep_pickup_datetime=datetime.datetime(2016, 2, 13, 21, 47, 53, tzinfo=<StaticTzInfo 'Etc/UTC'>), tpep_dropoff_datetime=datetime.datetime(2016, 2, 13, 21, 57, 15, tzinfo=<StaticTzInfo 'Etc/UTC'>), trip_distance=1.4, fare_amount=8.0, pickup_zip=10103, dropoff_zip=10110)
Row(tpep_pickup_datetime=datetime.datetime(2016, 2, 13, 18, 29, 9, tzinfo=<StaticTzInfo 'Etc/UTC'>), tpep_dropoff_datetime=datetime.datetime(2016, 2, 13, 18, 37, 23, tzinfo=<StaticTzInfo 'Etc/UTC'>), trip_distance=1.31, fare_amount=7.5, pickup_zip=10023, dropoff_zip=10023)


In [6]:
# Statement-level tags:
from databricks import sql
import os

with sql.connect(
    server_hostname = os.getenv("DATABRICKS_HOST"),
    http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
    access_token    = os.getenv("DATABRICKS_TOKEN"),
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT * FROM samples.nyctaxi.trips LIMIT ?",
            parameters=[2],
            query_tags={"team": "engineering", "dashboard": "abc123", "env": "prod"}
        )
        result = cursor.fetchall()
        for row in result:
            print(row)

Row(tpep_pickup_datetime=datetime.datetime(2016, 2, 13, 21, 47, 53, tzinfo=<StaticTzInfo 'Etc/UTC'>), tpep_dropoff_datetime=datetime.datetime(2016, 2, 13, 21, 57, 15, tzinfo=<StaticTzInfo 'Etc/UTC'>), trip_distance=1.4, fare_amount=8.0, pickup_zip=10103, dropoff_zip=10110)
Row(tpep_pickup_datetime=datetime.datetime(2016, 2, 13, 18, 29, 9, tzinfo=<StaticTzInfo 'Etc/UTC'>), tpep_dropoff_datetime=datetime.datetime(2016, 2, 13, 18, 37, 23, tzinfo=<StaticTzInfo 'Etc/UTC'>), trip_distance=1.31, fare_amount=7.5, pickup_zip=10023, dropoff_zip=10023)


Insert data

In [8]:
from databricks import sql
import os

with sql.connect(server_hostname = os.getenv("DATABRICKS_HOST"),
                 http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
                 access_token    = os.getenv("DATABRICKS_TOKEN")) as connection:

  with connection.cursor() as cursor:
    #cursor.execute("CREATE TABLE IF NOT EXISTS squares (x int, x_squared int)")

    #squares = [(i, i * i) for i in range(100)]

    #cursor.executemany("INSERT INTO squares VALUES (?, ?)", squares)

    cursor.execute("SELECT * FROM squares LIMIT ?", [10])

    result = cursor.fetchall()

    for row in result:
      print(row)

Row(x=39, x_squared=1521)
Row(x=43, x_squared=1849)
Row(x=0, x_squared=0)
Row(x=14, x_squared=196)
Row(x=28, x_squared=784)
Row(x=23, x_squared=529)
Row(x=32, x_squared=1024)
Row(x=25, x_squared=625)
Row(x=18, x_squared=324)
Row(x=10, x_squared=100)


Query metadata

In [9]:
from databricks import sql
import os

with sql.connect(server_hostname = os.getenv("DATABRICKS_HOST"),
                 http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
                 access_token    = os.getenv("DATABRICKS_TOKEN")) as connection:

  with connection.cursor() as cursor:
    cursor.columns(schema_name="default", table_name="squares")
    print(cursor.fetchall())

[Row(TABLE_CAT='workspace', TABLE_SCHEM='default', TABLE_NAME='squares', COLUMN_NAME='x', DATA_TYPE=4, TYPE_NAME='INT', COLUMN_SIZE=4, BUFFER_LENGTH=None, DECIMAL_DIGITS=0, NUM_PREC_RADIX=10, NULLABLE=1, REMARKS='', COLUMN_DEF=None, SQL_DATA_TYPE=None, SQL_DATETIME_SUB=None, CHAR_OCTET_LENGTH=None, ORDINAL_POSITION=0, IS_NULLABLE='YES', SCOPE_CATALOG=None, SCOPE_SCHEMA=None, SCOPE_TABLE=None, SOURCE_DATA_TYPE=None, IS_AUTO_INCREMENT='NO'), Row(TABLE_CAT='workspace', TABLE_SCHEM='default', TABLE_NAME='squares', COLUMN_NAME='x_squared', DATA_TYPE=4, TYPE_NAME='INT', COLUMN_SIZE=4, BUFFER_LENGTH=None, DECIMAL_DIGITS=0, NUM_PREC_RADIX=10, NULLABLE=1, REMARKS='', COLUMN_DEF=None, SQL_DATA_TYPE=None, SQL_DATETIME_SUB=None, CHAR_OCTET_LENGTH=None, ORDINAL_POSITION=1, IS_NULLABLE='YES', SCOPE_CATALOG=None, SCOPE_SCHEMA=None, SCOPE_TABLE=None, SOURCE_DATA_TYPE=None, IS_AUTO_INCREMENT='NO')]


HTTP request failed after retries: HTTPSConnectionPool(host='unknown-host', port=443): Max retries exceeded with url: /telemetry-unauth (Caused by NameResolutionError("HTTPSConnection(host='unknown-host', port=443): Failed to resolve 'unknown-host' ([Errno -3] Temporary failure in name resolution)"))


#### Manage cursors and connections

It is a best practice to close any connections and cursors that are no longer in use. This frees resources on Databricks all-purpose compute and Databricks SQL warehouses.

You can use a context manager (the with syntax used in previous examples) to manage the resources, or explicitly call close:

In [10]:
from databricks import sql
import os

connection = sql.connect(server_hostname = os.getenv("DATABRICKS_HOST"),
                         http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
                         access_token    = os.getenv("DATABRICKS_TOKEN"))

cursor = connection.cursor()

cursor.execute("SELECT * from range(10)")
print(cursor.fetchall())

cursor.close()
connection.close()

[Row(id=0), Row(id=1), Row(id=2), Row(id=3), Row(id=4), Row(id=5), Row(id=6), Row(id=7), Row(id=8), Row(id=9)]


##### Manage files in Unity Catalog volumes

The Databricks SQL Connector enables you to write local files to Unity Catalog volumes, download files from volumes, and delete files from volumes, as shown in the following example:

In [12]:
from databricks import sql
import os

# For writing local files to volumes and downloading files from volumes,
# you must set the staging_allowed_local_path argument to the path to the
# local folder that contains the files to be written or downloaded.
# For deleting files in volumes, you must also specify the
# staging_allowed_local_path argument, but its value is ignored,
# so in that case its value can be set for example to an empty string.
with sql.connect(server_hostname            = os.getenv("DATABRICKS_HOST"),
                 http_path                  = os.getenv("DATABRICKS_HTTP_PATH"),
                 access_token               = os.getenv("DATABRICKS_TOKEN"),
                 staging_allowed_local_path = "volumes/tmp/") as connection:

  with connection.cursor() as cursor:

    # Write a local file to the specified path in a volume.
    # Specify OVERWRITE to overwrite any existing file in that path.
    # cursor.execute(
    #   "PUT '/tmp/my-data.csv' INTO '/Volumes/workspace/default/dbx_volume/employee.csv' OVERWRITE"
    # )

    # Download a file from the specified path in a volume.
    # cursor.execute(
    #   "GET '/Volumes/workspace/default/dbx_volume/employee.csv' TO 'volumes/tmp/employee.csv'"
    # )
    cursor.execute(
      "GET '/Volumes/workspace/default/dbx_volume/departments.csv' TO 'volumes/tmp/departments.csv'"
    )

    # Delete a file from the specified path in a volume.
    # cursor.execute(
    #   "REMOVE '/Volumes/workspace/default/dbx_volume/employee.csv'"
    # )

#### Configure logging
The Databricks SQL Connector uses Python's standard logging module. The following example configures the logging level and generates a debug log:

In [13]:
from databricks import sql
import os, logging

logging.getLogger("databricks.sql").setLevel(logging.DEBUG)
logging.basicConfig(filename = "results.log", level    = logging.DEBUG)

connection = sql.connect(server_hostname = os.getenv("DATABRICKS_HOST"),
                         http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
                         access_token    = os.getenv("DATABRICKS_TOKEN"))

cursor = connection.cursor()

cursor.execute("SELECT * from range(10)")

result = cursor.fetchall()

for row in result:
   logging.debug(row)

cursor.close()
connection.close()

#### 

In [ ]:

from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import SQLDatabaseToolkit

# 2. Initialize the Databricks SQL database connection wrapper
db = SQLDatabase.from_databricks(
    catalog="samples",
    schema="nyctaxi",
    host=DATABRICKS_HOST,
    api_token=DATABRICKS_TOKEN,
    warehouse_id=WAREHOUSE_ID,
    #include_tables=[]
)

# 3. Instantiate your LLM of choice
llm = ChatOpenAI(model="gemma4:latest", temperature=0)

# 4. Construct the toolkit and the agent executor
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
agent_executor = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,
    agent_type="openai-tools"
)

# 5. Run a natural language query
response = agent_executor.invoke({"input": "What was the average fare amount for trips in 2016 ?"})
print(response["output"])


[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


trips
Invoking: `sql_db_schema` with `{'table_names': 'trips'}`



CREATE TABLE trips (
	tpep_pickup_datetime TIMESTAMP, 
	tpep_dropoff_datetime TIMESTAMP, 
	trip_distance DOUBLE, 
	fare_amount DOUBLE, 
	pickup_zip INT, 
	dropoff_zip INT
) USING DELTA
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'enabled')

/*
3 rows from trips table:
tpep_pickup_datetime	tpep_dropoff_datetime	trip_distance	fare_amount	pickup_zip	dropoff_zip
2016-02-13 21:47:53+00:00	2016-02-13 21:57:15+00:00	1.4	8.0	10103	10110
2016-02-13 18:29:09+00:00	2016-02-13 18:37:23+00:00	1.31	7.5	10023	10023
2016-02-06 19:40:58+00:00	2016-02-06 19:52:32+00:00	1.8	9.5	10001	10018
*/
Invoking: `sql_db_query_checker` with `{'query': "SELECT AVG(fare_amount) FROM trips WHERE STRFTIME('%Y', tpep_pickup_datetime) = '2016'"}`


SELECT AVG(fare_amount) FROM trips WHERE YEAR(tpep_pickup_datetime) = 2016
Invoking: `sql_db_quer

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks, DatabricksMCPServer, DatabricksMultiServerMCPClient
from langchain.agents import create_agent
from com.example.ai.LLMManager import LLMManager
 
workspace_client = WorkspaceClient()
host = workspace_client.config.host

mcp_client = DatabricksMultiServerMCPClient([
    DatabricksMCPServer(
        name="uc-functions",
        url=f"{host}/api/2.0/mcp/functions/workspace/default",
        workspace_client=workspace_client,
    ),
])

async with mcp_client:
    tools = await mcp_client.get_tools()
    agent = create_agent(
        ChatDatabricks(endpoint="databricks-claude-sonnet-4-5"),
        tools=tools,
    )
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "Look up customer info for Acme Corp"}]}
    )
    print(result["messages"][-1].content)



NotImplementedError: As of langchain-mcp-adapters 0.1.0, MultiServerMCPClient cannot be used as a context manager (e.g., async with MultiServerMCPClient(...)). Instead, you can do one of the following:
1. client = MultiServerMCPClient(...)
   tools = await client.get_tools()
2. client = MultiServerMCPClient(...)
   async with client.session(server_name) as session:
       tools = await load_mcp_tools(session)

In [ ]:
import asyncio

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
#from langchain_mcp_adapters.tools import convert_mcp_tool # Or pass via MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

async def main():
    # 1. Initialize the official Databricks Workspace SDK
    workspace_client = WorkspaceClient()
    host = workspace_client.config.host

    # 2. Point to your specific Databricks MCP Server URL
    # (Example shows a Managed Unity Catalog function endpoint)
    server_url = f"{host}/api/2.0/mcp/functions/workspace/default"
    
    mcp_client = DatabricksMCPClient(
        server_url=server_url,
        workspace_client=workspace_client,
    )

    # 3. Pull the live schema tools from Databricks
    # Databricks native client returns tools which can be mapped directly
    mcp_tools = mcp_client.list_tools()
    
    # 4. Convert the fetched MCP tools to LangChain/LangGraph style
    # Alternatively, you can feed the server params into MultiServerMCPClient 
    from langchain_mcp_adapters.client import MultiServerMCPClient
    
    # Using LangChain's MultiServer client via an SSE/HTTP link configuration:
    async with MultiServerMCPClient(
        {
            "databricks_tools": {
                "url": f"{server_url}/sse", # Databricks applications expose tools over SSE
                "transport": "sse",
                "headers": {"Authorization": f"Bearer {workspace_client.config.token}"}
            }
        }
    ) as langchain_mcp_client:
        
        # 5. Extract the tools
        langchain_tools = langchain_mcp_client.get_tools()

        # 6. Bind them to your LangChain Agent workflow
        model = ChatOpenAI(model="gpt-4o")
        agent = create_react_agent(model, langchain_tools)

        # 7. Query your agent 
        response = await agent.ainvoke(
            {"messages": [("user", "Run the calculation using the Unity Catalog function.")]}
        )
        print(response["messages"][-1].content)

if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
from databricks_langchain import UCFunctionToolkit
#from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent
from com.example.ai.LLMManager import LLMManager

# Pass the absolute path of your function directly into LangChain
toolkit = UCFunctionToolkit(function_names=["workspace.default.add_numbers"])
langchain_tools = toolkit.tools

model = LLMManager.get_model()

# Deploy instantly inside your agent structure
agent = create_agent(model, langchain_tools)


In [ ]:
import asyncio
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
from langchain_core.tools import tool, BaseTool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from databricks_langchain import ChatDatabricks

async def main():
    # 1. Initialize Databricks authenticated workspace client
    ws_client = WorkspaceClient()
    
    # 2. Define your Databricks Managed or Custom MCP Server URL
    # Replace the parameters below matching your catalog and schema details
    mcp_server_url = f"{ws_client.config.host}/api/2.0/mcp/functions/my_catalog/my_schema"
    
    print(f"Connecting to Databricks MCP Server at: {mcp_server_url}")
    
    # 3. Instantiate the Databricks MCP Client
    mcp_client = DatabricksMCPClient(
        server_url=mcp_server_url,
        client=ws_client
    )
    
    # 4. Asynchronously fetch available tools from the server
    raw_mcp_tools = await mcp_client.alist_tools()
    
    # 5. Adapt the raw MCP tools into standard LangChain-compatible tools
    langchain_tools = []
    for mcp_tool in raw_mcp_tools:
        
        # Create a dynamic function to invoke the specific tool
        async def make_tool_call(arguments: dict, name=mcp_tool.name):
            result = await mcp_client.acall_tool(tool_name=name, arguments=arguments)
            # Combine content components into a single text output string
            return "\n".join([c.text for c in result.content if hasattr(c, 'text')])

        # Wrap it inside a LangChain structured/functional tool block
        lc_tool = tool(
            name=mcp_tool.name, 
            description=mcp_tool.description
        )(make_tool_call)
        
        langchain_tools.append(lc_tool)
        print(f"Successfully loaded tool: {mcp_tool.name}")
        
    # 6. Initialize an LLM via Databricks Model Serving
    llm = ChatDatabricks(endpoint="databricks-claude-sonnet-4-5")
    
    # 7. Create a standard LangGraph / LangChain tool executor agent
    agent = create_react_agent(llm, tools=langchain_tools)
    
    # 8. Test the execution query
    response = await agent.ainvoke({
        "messages": [HumanMessage(content="Query the system metrics from the MCP tool.")]
    })
    
    print("\nAgent Output:")
    print(response["messages"][-1].content)

if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
import asyncio
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from databricks.sdk import WorkspaceClient
from langchain_openai import ChatOpenAI

# 1. Fetch your Databricks endpoint URL for SQL/Genie
# Format: https://<workspace-host>/api/2.0/mcp/functions/<catalog>/<schema>
DATABRICKS_HOST = "https://your-workspace.cloud.databricks.com"
SERVER_URL = f"{DATABRICKS_HOST}/api/2.0/mcp/functions/main/default"

async def main():
    # 2. Extract active authenticated session headers using Databricks SDK
    w = WorkspaceClient()
    headers = w.config.authenticate()

    # 3. Connect to the Databricks SQL MCP Server
    async with MultiServerMCPClient() as client:
        await client.add_server(
            "databricks-sql",
            url=SERVER_URL,
            headers=headers
        )
        
        # 4. Read discovered SQL functions from the server as LangChain tools
        tools = client.get_tools()
        
        # 5. Bind the tools directly to your preferred LLM
        model = ChatOpenAI(model="gpt-4o").bind_tools(tools)
        
        # 6. Query your Databricks environment
        query = "Show me the total sales row count from the 'orders' table."
        response = model.invoke([HumanMessage(content=query)])
        
        print("LLM Response:", response.content)
        print("Suggested Tool Calls:", response.tool_calls)

if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
import asyncio
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from databricks_langchain import ChatDatabricks
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client as connect

# Databricks workspace configs
DATABRICKS_HOST = "https://your-workspace-url.cloud.databricks.com"
# For managed Databricks SQL servers or your custom app endpoint
MCP_SERVER_URL = f"{DATABRICKS_HOST}/api/2.0/mcp/functions/your_catalog/your_schema"
DATABRICKS_TOKEN = "dapi..." 

async def main():
    # 1. Establish headers for Databricks Authentication
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json"
    }

    # 2. Connect to the Databricks MCP Server
    async with connect(MCP_SERVER_URL, headers=headers) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            # Initialize the session protocol
            await session.initialize()
            
            # List available tools from the server
            mcp_tools = await session.list_tools()
            print(f"Discovered MCP Tools: {[t.name for t in mcp_tools.tools]}")

            # 3. Wrap the Databricks execute_sql MCP tool into a LangChain tool
            @tool
            async def execute_sql_tool(query: str) -> str:
                """Executes a SQL query against Databricks tables in Unity Catalog."""
                # Call the underlying MCP server tool directly
                result = await session.call_tool("execute_sql", arguments={"query": query})
                return str(result.content)

            # 4. Initialize your LangChain Chat Model (e.g., ChatDatabricks or Claude)
            # Make sure your endpoint is provisioned in Databricks Model Serving
            llm = ChatDatabricks(endpoint="databricks-meta-llama-3-1-70b-instruct")

            # 5. Compile the LangChain tool calling Agent
            agent = create_react_agent(llm, tools=[execute_sql_tool])

            # 6. Execute a query through the agent
            query_prompt = "Find the top 5 records from the gold_sales_table ordered by transaction_date."
            response = await agent.ainvoke({
                "messages": [HumanMessage(content=query_prompt)]
            })

            # Output the agent's logic and final summarized answer
            for message in response["messages"]:
                print(f"[{message.type.upper()}]: {message.content}\n")

if __name__ == "__main__":
    asyncio.run(main())
